# Lab04

Stanisław Nieradko 193044

In [45]:
from pyspark import SparkContext
import itertools
import sys
import time

try:
    sc = SparkContext(appName="Lab04")
    sc.setLogLevel("ERROR")
except:
    pass

print(f"Spark version: {sc.version}")

Spark version: 4.1.1


In [46]:
def split_line(line):
    items = line.split(' ')
    while '' in items:
        items.remove('')
    return items
file = sc.textFile('datasets/4.txt')
lines = file.map(lambda line: split_line(line)).cache()

print(f"Total: {lines.count()}")
for i in lines.take(5):
    print(i)

Total: 31101
['FRO11987', 'ELE17451', 'ELE89019', 'SNA90258', 'GRO99222']
['GRO99222', 'GRO12298', 'FRO12685', 'ELE91550', 'SNA11465', 'ELE26917', 'ELE52966', 'FRO90334', 'SNA30755', 'ELE17451', 'FRO84225', 'SNA80192']
['ELE17451', 'GRO73461', 'DAI22896', 'SNA99873', 'FRO86643']
['ELE17451', 'ELE37798', 'FRO86643', 'GRO56989', 'ELE23393', 'SNA11465']
['ELE17451', 'SNA69641', 'FRO86643', 'FRO78087', 'SNA11465', 'GRO39357', 'ELE28573', 'ELE11375', 'DAI54444']


## Singles

In [47]:
start_time = time.time()

singles = lines.flatMap(lambda x: x).map(lambda x: ((x,), 1))
singles = singles.reduceByKey(lambda x, y: x + y)
singles_filtered = singles.filter(lambda x: x[1] >= 100)
frequent_singles = singles_filtered.collect()
frequent_items_set = set([item[0][0] for item in frequent_singles])

print(f"Singles: {len(frequent_items_set)}.")
for i in frequent_singles[:5]:
    print(f"{i[0][0]}: {i[1]}")

Singles: 647.
ELE17451: 3875
GRO99222: 906
GRO12298: 385
ELE52966: 380
SNA30755: 456


## Pairs

In [48]:
def get_frequent_pairs(session):
    filtered = sorted([item for item in session if item in frequent_items_set])
    return itertools.combinations(filtered, 2)

pairs = lines.flatMap(get_frequent_pairs).map(lambda p: (p, 1))
pairs = pairs.reduceByKey(lambda a, b: a + b)
frequent_pairs = pairs.filter(lambda x: x[1] >= 100).sortBy(lambda x: x[1], ascending=False)
frequent_pairs_results = frequent_pairs.collect()

print(f"Pairs: {len(frequent_pairs_results)}.")
for p in frequent_pairs_results[:10]:
    print(f"{p[0][0]} [{p[0][1]}]: {p[1]}")

Pairs: 1334.
DAI62779 [ELE17451]: 1592
FRO40251 [SNA80324]: 1412
DAI75645 [FRO40251]: 1254
FRO40251 [GRO85051]: 1213
DAI62779 [GRO73461]: 1139
DAI75645 [SNA80324]: 1130
DAI62779 [FRO40251]: 1070
DAI62779 [SNA80324]: 923
DAI62779 [DAI85309]: 918
ELE32164 [GRO59710]: 911


## Triples

In [49]:
def get_frequent_triples(session):
    filtered = sorted([item for item in session if item in frequent_items_set])
    return itertools.combinations(filtered, 3)

triples = lines.flatMap(get_frequent_triples).map(lambda t: (t, 1))
triples = triples.reduceByKey(lambda a, b: a + b)
frequent_triples = triples.filter(lambda x: x[1] >= 100).sortBy(lambda x: x[1], ascending=False)
frequent_triples_results = frequent_triples.collect()

print(f"Triples: {len(frequent_triples_results)}.")
for t in frequent_triples_results[:15]:
    print(f"{t[0][0]} {t[0][1]} [{t[0][2]}]: {t[1]}")

Triples: 233.
DAI75645 FRO40251 [SNA80324]: 550
DAI62779 FRO40251 [SNA80324]: 476
FRO40251 GRO85051 [SNA80324]: 471
DAI62779 ELE92920 [SNA18336]: 432
DAI62779 DAI75645 [SNA80324]: 421
DAI62779 ELE17451 [SNA80324]: 417
DAI62779 DAI75645 [FRO40251]: 412
DAI62779 ELE17451 [FRO40251]: 406
DAI75645 FRO40251 [GRO85051]: 395
DAI62779 FRO40251 [GRO85051]: 381
ELE17451 FRO40251 [SNA80324]: 353
DAI62779 ELE17451 [ELE92920]: 345
FRO40251 FRO92469 [SNA80324]: 343
DAI62779 DAI85309 [ELE17451]: 339
DAI62779 DAI75645 [ELE17451]: 328


In [50]:
total_time = time.time() - start_time
sc.stop()

In [51]:
from IPython.display import display, Markdown

summary_text = f"""
## Podsumowanie

W ramach laboratorium zaimplementowano algorytm **A-priori** wykorzystując model programistyczny RDD w Apache Spark. Dzięki równoległemu przetwarzaniu danych możliwe było efektywne przeliczenie wystąpień tysięcy kombinacji produktów.

### Kluczowe wyniki:
- **Singles:** {len(frequent_items_set)}
- **Pairs:** {len(frequent_pairs_results)}
- **Triples:** {len(frequent_triples_results)}
- **Czas wykonania:** {total_time:.2f}s

Najczęstsza para to `{frequent_pairs_results[0][0][0]}` i `{frequent_pairs_results[0][0][1]}` ({frequent_pairs_results[0][1]} wystąpienia), natomiast najczęstsza trójka to `{frequent_triples_results[0][0][0]}`, `{frequent_triples_results[0][0][1]}` oraz `{frequent_triples_results[0][0][2]}` ({frequent_triples_results[0][1]} wystąpień). Algorytm ten pozwala na odkrywanie ukrytych zależności w koszykach zakupowych, co może być wykorzystane w systemach rekomendacyjnych lub planowaniu ekspozycji towarów.
"""

display(Markdown(summary_text))


## Podsumowanie

W ramach laboratorium zaimplementowano algorytm **A-priori** wykorzystując model programistyczny RDD w Apache Spark. Dzięki równoległemu przetwarzaniu danych możliwe było efektywne przeliczenie wystąpień tysięcy kombinacji produktów.

### Kluczowe wyniki:
- **Singles:** 647
- **Pairs:** 1334
- **Triples:** 233
- **Czas wykonania:** 9.96s

Najczęstsza para to `DAI62779` i `ELE17451` (1592 wystąpienia), natomiast najczęstsza trójka to `DAI75645`, `FRO40251` oraz `SNA80324` (550 wystąpień). Algorytm ten pozwala na odkrywanie ukrytych zależności w koszykach zakupowych, co może być wykorzystane w systemach rekomendacyjnych lub planowaniu ekspozycji towarów.
